In [ ]:
# 1、引入sft trainer
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "logs/Qwen3-0.6B-SFT"

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0402 13:52:47.339000 44628 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# 2、加载数据集
from datasets import load_dataset
dataset = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

In [3]:
dataset["train"][0]

{'conversation_id': 501,
 'category': 'dialogue',
 'conversation': [{'human': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'assistant': '高氟铍矿;配料;熔炼;回收率;脱氟率'}],
 'dataset': 'psychology'}

In [3]:
# 3、定义一个map函数，将数据集中的每个样本，转换成SFTTrainer支持的格式
def map_to_sft_format(example):
    conversation = example["conversation"]
    message_list = []
    for conv in conversation:
        for key ,value in conv.items():
            key = "user" if key == "human" else key
            message_list.append({"role":key,"content":value})

    return {"messages":message_list}

In [4]:
# 4、对dataset 进行转换
remove_list = list(dataset["train"][0].keys())
dataset = dataset.map(map_to_sft_format,batched=False,remove_columns=remove_list)

In [6]:
dataset["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [ ]:
# 5、构造SFTConfig对象
config = SFTConfig(
    output_dir = "finetuned/Qwen3-0.6B-trl-sft",
    per_device_train_batch_size = 3,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-5,
    max_steps = 3000,
    # 日志
    logging_steps = 100,
    
    report_to = ["tensorboard"],
    # 显存优化相关:
    bf16=True, # 混合精度
    gradient_checkpointing=True, # 梯度检查点
    activation_offloading = True, # CPU 卸载
    # 保存相关
    save_strategy = "steps",
    save_steps = 300,
    # 评估相关
    eval_steps = 300,
    eval_strategy = "steps",
    metric_for_best_model = "eval_loss",
    load_best_model_at_end=True,
    greater_is_better = False,
    max_length = 2500,
    
    chat_template_path="HuggingFaceTB/SmolLM3-3B",
    assistant_only_loss=True
)

In [ ]:
# 6、构造SFTTrainer对象
from transformers import AutoModelForCausalLM,AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B-Base")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B-Base")

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    args = config,
    train_dataset= dataset["train"],
    eval_dataset=dataset["test"],
    
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 955.24it/s, Materializing param=model.norm.weight]                               


In [7]:
# 7、验证数据集是否已经是chat template形式
data_loader = trainer.get_train_dataloader()
for batch in data_loader:
    res = tokenizer.decode(batch["input_ids"])
    print(res[0])
    break

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 02 April 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
[RANK 0] curr_pct=83.5% > self.virtual_memory_safe_pct=60% of virtual memory used


Step,Training Loss,Validation Loss


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [8]:
trainer.save_model("./finetuned/Qwen3-0.6B-SFT-best")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it]


In [9]:
from transformers import AutoTokenizer
tokenzier = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

In [10]:
tokenizer.save_pretrained("model/HuggingFaceTB-SmolLM3-3B")

('model/HuggingFaceTB-SmolLM3-3B\\tokenizer_config.json',
 'model/HuggingFaceTB-SmolLM3-3B\\chat_template.jinja',
 'model/HuggingFaceTB-SmolLM3-3B\\tokenizer.json')

In [ ]:
from peft import prepare_model_for_kbit_training